# CRIS Section 3 - Competitive Rankers

Four queries, comments removed.

## v74

From the two diagnostics run on 27 Sep 2026 against Aya's review.

**2.13 takes `p_norm_basis`** (`[S2-DPNORM]`), default `REPEATS`: the daypart
average is built on repeat half hours only, which reproduces Jill's 128 for
HIST M-SU 8P-12A exactly. `ALL` (premieres and repeats) is what v73 and earlier
did and gives 164. The output carries `DAYPART_NORM_BASIS`. ⚠ changes 2.13's
DAYPART_AVG_000 wherever the daypart has premieres in it.

**2.2 on Jill's window** (9/14/25-9/13/26) returns the premiere norm on 393
telecasts / 32 series, series 221, norm 219. No query change; the 394 CRIS
showed is the default window one week later.

No table change. Requires `CRIS_Tables_v73` (renamed v74 here, unchanged).

## v73

From Aya's review of 27 Sep 2026.

**Every query caps its window end at the latest full Sunday with data**
(`[CAP-SUN]`). A later date passed in, such as a fiscal-year end computed by the
planner, no longer prints as TO_DATE on a window that has no data there: SP_4
showed TO_DATE 2026-09-27 on data that ends 2026-09-20. Section 1 caps the
target night at the latest night loaded. 3.2, 3.3 and 3.4 read that Sunday from
`CRIS_LATEST_DATE` instead of scanning the fact table for MAX(BROADCAST_DATE).

**2.11** adds the norm's own movement next to the series' movement:
`NORM_VS_PRIOR_QH_PCT` and `NORM_LAST_VS_FIRST_PCT`, so a QH1 to QH4 drift can
be read against how the time period itself drifts (A&E answer template #15).

No table change. Requires `CRIS_Tables_v72` (renamed v73 here, unchanged).

## v72

From the CRIS UI test pass of 25 Sep 2026.

**2.10** finds a lead-in that ends within 5 minutes of the block start
(`p_adjacent_min`), not only on the exact second: Nielsen books the 8pm
Skinwalker repeat as ending 20:58 because a two-minute short inside it is
carved out, and exact adjacency left five nights with no lead-in. A night with
no lead-in now reads NO LEAD-IN, not PREMIERE.

**2.5, 2.6** report DEMO_USED as P2: median age and skew are measured on the
whole audience, not on the network's key demo.

**2.7** carries a neutral network default; it still held LIF from the v62 case
set, and a fan-out that failed to pass the network fell to it.

**2.11** closes its window on the last full Sunday, as every 12-month norm does.

Requires `CRIS_Tables_v71` (renamed v72 here, unchanged).


**3.2 uses DENSE_RANK**, so tied networks share a rank. R_6 returned 11 where A&E
print "10 (tied)".

**3.3 no longer returns the competitive-set rank.** A ranker asked for within a
network reports the rank within that network. It is still computed, because
`p_top_n` cuts on it.

**3.4 reads `CRIS_TELECAST_FACT`.** `CRIS_TELECAST_ALL_NET` is gone: with the
six-network filter removed the two tables held the same rows.


## 3.1 Series Rank, Network Time Period

**A&E case:** R_3: where AEN ranks among entertainment cable in INTERROGATION RAW's time period, 7/2/26

**Expected:** TBS 475 at #1, AEN 258 at #2


In [ ]:
%%sql
SET p_network       = 'AEN';
SET p_program       = '%INTERROGATION RAW%';
SET p_season        = NULL;
SET p_stream        = 'Live+SD';
SET p_demo          = 'P25_64_AA_ESTIMATES';
SET p_excl_by       = 'CODE';
SET p_include_specials = FALSE;
SET p_win_from      = '2026-07-02';
SET p_win_to        = '2026-07-02';
SET p_slot_min_min  = 15;

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH season_pick AS (
    SELECT CT_SEASON
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    WHERE NETWORK_CODE  = $p_network
      AND RATING_SOURCE = $p_stream
      AND IS_FIRST_RUN_CT
      AND PROGRAM_NAME ILIKE $p_program
      AND CT_SEASON IS NOT NULL
      AND ($p_include_specials OR CONTENT_TYPE <> 'Special')
    QUALIFY ROW_NUMBER() OVER (ORDER BY BROADCAST_DATE DESC) = 1
),
slots AS (
    SELECT h.BROADCAST_DATE, h.HALFHR_START_TIME,
           SUM(h.HH_MIN) AS SERIES_MIN
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT h
    WHERE h.NETWORK_CODE  = $p_network
      AND h.RATING_SOURCE = $p_stream
      AND h.IS_FIRST_RUN_CT
      AND h.PROGRAM_NAME ILIKE $p_program
      AND h.CT_SEASON IS NOT NULL
      AND ($p_include_specials OR h.CONTENT_TYPE <> 'Special')
      AND ( $p_season IS NOT NULL AND h.CT_SEASON = $p_season
         OR $p_season IS NULL AND h.CT_SEASON = (SELECT CT_SEASON FROM season_pick) )
      AND ($p_win_from IS NULL OR h.BROADCAST_DATE >= TO_DATE($p_win_from))
      AND ($p_win_to   IS NULL OR h.BROADCAST_DATE <= TO_DATE($p_win_to))
    GROUP BY h.BROADCAST_DATE, h.HALFHR_START_TIME
    HAVING ($p_slot_min_min IS NULL OR SUM(h.HH_MIN) >= $p_slot_min_min)
),
competitive AS (
    SELECT h.NETWORK_CODE,
           COUNT(DISTINCT h.BROADCAST_DATE)               AS N_DATES,
           COUNT(*)                                       AS N_HALFHRS,
           SUM(h.HH_MIN)                                  AS MINUTES,
           SUM(IDENTIFIER($p_demo    ) * h.HH_MIN)
             / NULLIF(SUM(h.HH_MIN), 0)                   AS DEMO_AA
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT h
    JOIN slots s
      ON  s.BROADCAST_DATE    = h.BROADCAST_DATE
      AND s.HALFHR_START_TIME = h.HALFHR_START_TIME
    JOIN AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE u
      ON u.NETWORK_CODE = h.NETWORK_CODE
    WHERE h.RATING_SOURCE = $p_stream
      AND ( ($p_excl_by = 'CODE' AND u.IS_ENT_BY_CODE)
         OR ($p_excl_by = 'NAME' AND u.IS_ENT_BY_NAME) )
    GROUP BY h.NETWORK_CODE
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    RANK() OVER (ORDER BY ROUND(DEMO_AA/1000, 0) DESC)     AS RNK,
    COUNT(*) OVER ()                                       AS UNIVERSE_SIZE,
    NETWORK_CODE                                           AS NETWORK,
    IFF(NETWORK_CODE = $p_network, 1, 0)                     AS IS_TARGET,
    IFF(NETWORK_CODE = $p_network, '<<< the series network', NULL) AS MARKER,
    N_DATES, N_HALFHRS, MINUTES,
    ROUND(DEMO_AA/1000, 0)                                 AS AA_000,
    $p_demo                                                AS DEMO,
    $p_stream                                              AS STREAM,
    $p_excl_by                                             AS EXCLUSION_LIST,
    (SELECT COUNT(*) FROM slots)                           AS SLOTS,
    (SELECT MIN(SERIES_MIN) FROM slots)                    AS SLOT_MIN_MINUTES,
    (SELECT SUM(SERIES_MIN) FROM slots)                    AS SERIES_MINUTES,
    (SELECT MIN(BROADCAST_DATE) FROM slots)                AS PERIOD_FROM,
    (SELECT MAX(BROADCAST_DATE) FROM slots)                AS PERIOD_TO
FROM competitive
ORDER BY RNK;

## 3.2 Network Ranks

**A&E case:** R_6: HIST rank vs ad-supported entertainment cable, Prime and Total Day, FY26 through 7/5/26, Live+SD

**Expected:** Prime 184 at #7 of 100; Total Day 93 at #12


In [ ]:
%%sql
SET p_network       = 'HIST';
SET p_fiscal_year   = 2026;
SET p_fy_to         = '2026-07-05';
SET p_stream        = 'Live+SD';
SET p_demo          = 'P25_64_AA_ESTIMATES';
SET p_weight_col    = 'HH_MIN';
SET p_excl_by       = 'CODE';
SET data_max = (SELECT MAX(LATEST_FULL_SUNDAY) FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                WHERE RATING_SOURCE = $p_stream);
SET fy = (SELECT COALESCE($p_fiscal_year,
            (SELECT MAX(FISCAL_YEAR) FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
             WHERE FY_START <= $data_max)));
SET fy_start   = (SELECT FY_START FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR WHERE FISCAL_YEAR = $fy);
SET fy_to      = (SELECT LEAST(COALESCE(TO_DATE($p_fy_to), $data_max), COALESCE(
                    (SELECT FY_END FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR WHERE FISCAL_YEAR = $fy),
                    $data_max)));
SET pfy_start  = (SELECT PRIOR_FY_START FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR WHERE FISCAL_YEAR = $fy);
SET pfy_to     = (SELECT DATEADD('day', DATEDIFF('day', $fy_start, $fy_to), $pfy_start));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH dayparts AS (
              SELECT 1 AS ORD, 'PRIME M-SU 8P-11P' AS DAYPART, 2000 AS FROM_2959, 2300 AS TO_2959
    UNION ALL SELECT 2,         'TOTAL DAY 6A-6A',               600,               2960
),
periods AS (
              SELECT 'CURRENT'  AS PERIOD, $fy_start  AS D_FROM, $fy_to  AS D_TO
    UNION ALL SELECT 'YEAR AGO',           $pfy_start,           $pfy_to
),
measured AS (
    SELECT p.PERIOD, d.ORD, d.DAYPART, h.NETWORK_CODE,
           u.IS_CABLE, u.IS_ENT_BY_CODE, u.IS_ENT_BY_NAME,
           SUM(IDENTIFIER($p_demo    ) * COALESCE(IDENTIFIER($p_weight_col), h.HH_MIN))
             / NULLIF(SUM(COALESCE(IDENTIFIER($p_weight_col), h.HH_MIN)), 0) AS AA,
           SUM(h.HH_MIN) AS MINUTES
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT h
    JOIN AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE u
      ON u.NETWORK_CODE = h.NETWORK_CODE
    JOIN periods p
      ON h.BROADCAST_DATE BETWEEN p.D_FROM AND p.D_TO
    JOIN dayparts d
      ON h.HH_START_2959 >= d.FROM_2959 AND h.HH_START_2959 < d.TO_2959
    WHERE h.RATING_SOURCE = $p_stream
      AND u.IS_CABLE
      AND NOT u.IS_ARTIFACT
    GROUP BY p.PERIOD, d.ORD, d.DAYPART, h.NETWORK_CODE,
             u.IS_CABLE, u.IS_ENT_BY_CODE, u.IS_ENT_BY_NAME
),
ranked AS (
    SELECT m.*,
        DENSE_RANK() OVER (PARTITION BY PERIOD, ORD
                     ORDER BY ROUND(AA/1000, 0) DESC)          AS RNK_ALL_CABLE,
        DENSE_RANK() OVER (PARTITION BY PERIOD, ORD,
                     IFF( ($p_excl_by = 'CODE' AND IS_ENT_BY_CODE)
                       OR ($p_excl_by = 'NAME' AND IS_ENT_BY_NAME), 1, 0)
                     ORDER BY ROUND(AA/1000, 0) DESC)          AS RNK_ENT_CABLE,
        COUNT(*) OVER (PARTITION BY PERIOD, ORD,
                     IFF( ($p_excl_by = 'CODE' AND IS_ENT_BY_CODE)
                       OR ($p_excl_by = 'NAME' AND IS_ENT_BY_NAME), 1, 0),
                     ROUND(AA/1000, 0)) > 1                    AS TIED_ENT_CABLE,
        COUNT(*) OVER (PARTITION BY PERIOD, ORD, ROUND(AA/1000, 0)) > 1 AS TIED_ALL_CABLE,
        COUNT(*) OVER (PARTITION BY PERIOD, ORD)               AS N_ALL_CABLE,
        COUNT_IF( ($p_excl_by = 'CODE' AND IS_ENT_BY_CODE)
               OR ($p_excl_by = 'NAME' AND IS_ENT_BY_NAME) )
            OVER (PARTITION BY PERIOD, ORD)                     AS N_ENT_CABLE
    FROM measured m
),
net AS (
    SELECT * FROM ranked
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    c.DAYPART,
    c.NETWORK_CODE                                             AS NETWORK,
    $fy                                                        AS FISCAL_YEAR,
    $fy_start AS FY_FROM, $fy_to AS FY_TO,
    $pfy_start AS PRIOR_FROM, $pfy_to AS PRIOR_TO,
    ROUND(c.AA/1000, 0)                                        AS AA_000,
    IFF($p_network IS NOT NULL AND c.NETWORK_CODE = $p_network, 1, 0) AS IS_TARGET,
    c.RNK_ALL_CABLE                                            AS RANK_ALL_CABLE,
    c.TIED_ALL_CABLE                                           AS TIED_ALL_CABLE,
    c.RNK_ALL_CABLE || IFF(c.TIED_ALL_CABLE, ' (tied)', '')    AS RANK_ALL_CABLE_LABEL,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME), TRUE, FALSE)  AS IS_ENT_CABLE,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME), c.RNK_ENT_CABLE, NULL) AS RANK_ENT_CABLE,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME), c.TIED_ENT_CABLE, NULL) AS TIED,
    IFF( ($p_excl_by = 'CODE' AND c.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND c.IS_ENT_BY_NAME),
        c.RNK_ENT_CABLE || IFF(c.TIED_ENT_CABLE, ' (tied)', ''), NULL) AS RANK_ENT_CABLE_LABEL,
    ROUND(y.AA/1000, 0)                                        AS AA_000_YEAR_AGO,
    y.RNK_ALL_CABLE                                            AS RANK_ALL_CABLE_YEAR_AGO,
    y.RNK_ALL_CABLE || IFF(y.TIED_ALL_CABLE, ' (tied)', '')    AS RANK_ALL_CABLE_YEAR_AGO_LABEL,
    IFF( ($p_excl_by = 'CODE' AND y.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND y.IS_ENT_BY_NAME), y.RNK_ENT_CABLE, NULL) AS RANK_ENT_CABLE_YEAR_AGO,
    IFF( ($p_excl_by = 'CODE' AND y.IS_ENT_BY_CODE)
      OR ($p_excl_by = 'NAME' AND y.IS_ENT_BY_NAME),
        y.RNK_ENT_CABLE || IFF(y.TIED_ENT_CABLE, ' (tied)', ''), NULL) AS RANK_ENT_CABLE_YEAR_AGO_LABEL,
    ROUND((ROUND(c.AA/1000,0)
           / NULLIF(ROUND(y.AA/1000,0),0) - 1) * 100, 1)       AS PCT_VS_YEAR_AGO,
    c.N_ALL_CABLE                                              AS UNIVERSE_ALL_CABLE,
    c.N_ENT_CABLE                                              AS UNIVERSE_ENT_CABLE,
    $p_stream AS STREAM, $p_demo     AS DEMO,
    $p_weight_col AS WEIGHTED_ON, $p_excl_by AS EXCLUSION_LIST,
    $fy_to                  AS TO_DATE
FROM net c
LEFT JOIN net y ON y.NETWORK_CODE = c.NETWORK_CODE
               AND y.ORD = c.ORD AND y.PERIOD = 'YEAR AGO'
WHERE c.PERIOD = 'CURRENT'
ORDER BY c.ORD, c.RNK_ALL_CABLE, c.NETWORK_CODE;

## 3.3 Series Genre Ranker

**A&E case:** R_1: top 3 non-fiction shows per network, FY26 through 6/28/26, AEN HIST LIF

**Expected:** eight rows; AAs match A&E exactly


In [ ]:
%%sql
SET p_top_n         = NULL;
SET p_min_tcasts    = NULL;
SET p_focus_networks = 'AEN,HIST,LIF';
SET p_per_network    = 3;
SET p_genre_col     = 'CT_GENRE_1';
SET p_genre         = 'NON-FICTION';
SET p_genre_2_exclude = 'SPORTS';
SET p_stream        = 'Live+SD';
SET p_demo          = 'P25_64_AA_ESTIMATES';
SET p_excl_by       = 'CODE';
SET p_include_specials = FALSE;
SET p_premieres_only = TRUE;
SET p_dp_from_2959  = 2000;
SET p_dp_to_2959    = 2400;
SET p_fiscal_year   = 2026;
SET p_win_to        = '2026-06-28';
SET p_movies        = 'EXCLUDE';
SET data_max = (SELECT MAX(LATEST_FULL_SUNDAY) FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                WHERE RATING_SOURCE = $p_stream);
SET fy = (SELECT COALESCE($p_fiscal_year,
            (SELECT MAX(FISCAL_YEAR) FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
             WHERE FY_START <= $data_max)));
SET win_from = (SELECT FY_START FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR WHERE FISCAL_YEAR = $fy);
SET win_to   = (SELECT LEAST(COALESCE(TO_DATE($p_win_to), $data_max), $data_max));

SET demo_used = (SELECT UPPER(COALESCE($p_demo,
                     'DEMO_AA - resolves per network, unresolved here because this
                      query has no single network')));

WITH scoped AS (
    SELECT h.NETWORK_CODE, h.PROGRAM_CODE, h.PROGRAM_NAME,
           IDENTIFIER($p_genre_col)                            AS GENRE,
           h.TELECAST_NUM, h.HH_MIN,
           IDENTIFIER($p_demo    )                             AS DEMO
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT h
    JOIN AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE u
      ON u.NETWORK_CODE = h.NETWORK_CODE
    WHERE h.RATING_SOURCE = $p_stream
      AND h.BROADCAST_DATE BETWEEN $win_from AND $win_to
      AND h.HH_START_2959 >= $p_dp_from_2959
      AND h.HH_START_2959 <  $p_dp_to_2959
      AND ( ($p_excl_by = 'CODE' AND u.IS_ENT_BY_CODE)
         OR ($p_excl_by = 'NAME' AND u.IS_ENT_BY_NAME) )
      AND ($p_include_specials OR h.CONTENT_TYPE <> 'Special')
      AND (NOT $p_premieres_only OR h.IS_FIRST_RUN_CT)
      AND ($p_genre_2_exclude IS NULL
           OR h.CT_GENRE_2 IS NULL
           OR UPPER(h.CT_GENRE_2) NOT LIKE UPPER($p_genre_2_exclude))
      AND ( $p_movies = 'BOTH'
         OR ($p_movies = 'EXCLUDE' AND h.MOVIE_IND = 0)
         OR ($p_movies = 'ONLY'    AND h.MOVIE_IND = 1) )
),
per_telecast AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, TELECAST_NUM,
           MAX(PROGRAM_NAME) AS PROGRAM_NAME,
           MAX(GENRE)        AS GENRE,
           SUM(DEMO * HH_MIN) / NULLIF(SUM(HH_MIN), 0) AS TELECAST_AA,
           SUM(HH_MIN)       AS MINUTES
    FROM scoped
    GROUP BY NETWORK_CODE, PROGRAM_CODE, TELECAST_NUM
),
per_series AS (
    SELECT NETWORK_CODE, PROGRAM_CODE,
           MAX(PROGRAM_NAME)   AS PROGRAM_NAME,
           MAX(GENRE)          AS GENRE,
           COUNT(*)            AS TELECASTS,
           SUM(TELECAST_AA * MINUTES)
             / NULLIF(SUM(MINUTES), 0)  AS SERIES_AA,
           SUM(MINUTES)        AS MINUTES
    FROM per_telecast
    GROUP BY NETWORK_CODE, PROGRAM_CODE
    HAVING ($p_genre IS NULL OR MAX(GENRE) ILIKE $p_genre)
       AND ($p_min_tcasts IS NULL OR COUNT(*) >= $p_min_tcasts)
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    RANK() OVER (PARTITION BY NETWORK_CODE
                 ORDER BY ROUND(SERIES_AA/1000, 0) DESC)       AS RNK_IN_NETWORK,
    PROGRAM_NAME    AS PROGRAM,
    NETWORK_CODE    AS NETWORK,
    GENRE,
    TELECASTS,
    MINUTES,
    ROUND(SERIES_AA/1000, 0)                                   AS AA_000,
    $p_demo     AS DEMO, $p_stream AS STREAM,
    $win_from AS FROM_DATE, $win_to AS TO_DATE,
    $p_excl_by AS EXCLUSION_LIST
FROM per_series
QUALIFY ($p_top_n IS NULL
         OR RANK() OVER (ORDER BY ROUND(SERIES_AA/1000, 0) DESC) <= $p_top_n)
    AND ($p_focus_networks IS NULL
         OR ARRAY_CONTAINS(NETWORK_CODE::VARIANT, SPLIT($p_focus_networks, ',')))
    AND ($p_per_network IS NULL OR RNK_IN_NETWORK <= $p_per_network)
ORDER BY NETWORK, RNK_IN_NETWORK, AA_000 DESC, PROGRAM;

## 3.4 Telecast Ranker

**A&E case:** R_4: top 50 telecast ranker, Tuesday 7/14/26, P25+

**Expected:** FOX MLB 7611 at #1, AGT 4587


In [ ]:
%%sql
SET p_top_n         = 50;
SET p_universe      = 'TV';
SET p_stream        = 'Live+SD';
SET p_demo          = 'P25_AA_ESTIMATES';
SET p_dp_from_2959  = 2000;
SET p_dp_to_2959    = 2300;
SET p_win_from      = '2026-07-14';
SET p_win_to        = '2026-07-14';
SET data_max = (SELECT MAX(LATEST_FULL_SUNDAY) FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                WHERE RATING_SOURCE = $p_stream);
SET last_sun = $data_max;
SET win_to   = (SELECT LEAST(COALESCE(TO_DATE($p_win_to), $last_sun), $last_sun));
SET win_from = (SELECT COALESCE(TO_DATE($p_win_from), DATEADD('day', -6, $win_to)));

SET demo_used = (SELECT UPPER(COALESCE($p_demo,
                     'DEMO_AA - resolves per network, unresolved here because this
                      query has no single network')));

WITH in_daypart AS (
    SELECT DISTINCT NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
                    PROGRAM_CODE, TELECAST_NUM
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    WHERE RATING_SOURCE = $p_stream
      AND BROADCAST_DATE BETWEEN $win_from AND $win_to
      AND HH_START_2959 >= $p_dp_from_2959
      AND HH_START_2959 <  $p_dp_to_2959
),
telecasts AS (
    SELECT t.NETWORK_CODE, t.BROADCAST_DATE, t.DAY_OF_WEEK,
           t.PROGRAM_NAME, t.EPISODE_NAME,
           t.START_TIME, t.END_TIME, t.BCAST_START_2959,
           t.EVENT_DURATION, t.IS_FIRST_RUN_CT, t.IS_PREMIERE_CT,
           t.MEDIAN_AGE,
           IDENTIFIER($p_demo    )  AS DEMO,
           t.P2_AA_ESTIMATES, t.P25_64_AA_ESTIMATES,
           t.F2_AA_ESTIMATES, t.M2_AA_ESTIMATES
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT t
    JOIN in_daypart d
      ON  d.NETWORK_CODE   = t.NETWORK_CODE
      AND d.RATING_SOURCE  = t.RATING_SOURCE
      AND d.BROADCAST_DATE = t.BROADCAST_DATE
      AND d.PROGRAM_CODE   = t.PROGRAM_CODE
      AND d.TELECAST_NUM   = t.TELECAST_NUM
    JOIN AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE u
      ON u.NETWORK_CODE = t.NETWORK_CODE
    WHERE t.RATING_SOURCE = $p_stream
      AND t.BROADCAST_DATE BETWEEN $win_from AND $win_to
      AND NOT u.IS_ARTIFACT
      AND ( $p_universe = 'TV' OR ($p_universe = 'CABLE' AND u.IS_CABLE) )
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    RANK() OVER (ORDER BY ROUND(DEMO/1000, 0) DESC)            AS RNK,
    NETWORK_CODE    AS NETWORK,
    PROGRAM_NAME    AS PROGRAM,
    EPISODE_NAME    AS EPISODE,
    BROADCAST_DATE  AS AIR_DATE,
    DAY_OF_WEEK     AS DAY,
    START_TIME, END_TIME, EVENT_DURATION,
    IFF(NOT IS_FIRST_RUN_CT, 'R', 'P')                              AS PREMIERE_REPEAT,
    IFF(IS_PREMIERE_CT = 1, 'SEASON PREMIERE', NULL)       AS SEASON_PREMIERE,
    ROUND(DEMO/1000, 0)                                        AS AA_000,
    RANK() OVER (ORDER BY ROUND(P25_64_AA_ESTIMATES/1000, 0) DESC) AS RNK_P25_64,
    ROUND(P25_64_AA_ESTIMATES/1000, 0)                         AS P25_64_000,
    RANK() OVER (ORDER BY ROUND(P2_AA_ESTIMATES/1000, 0) DESC) AS RNK_P2,
    ROUND(P2_AA_ESTIMATES/1000, 0)                             AS P2_000,
    ROUND(MEDIAN_AGE, 0)                                       AS MEDIAN_AGE,
    ROUND(100.0 * F2_AA_ESTIMATES
          / NULLIF(P2_AA_ESTIMATES, 0), 0)                     AS FEMALE_PCT,
    $p_universe AS UNIVERSE, $p_stream AS STREAM,
    $win_from AS WEEK_FROM, $win_to AS WEEK_TO,
    $win_from               AS FROM_DATE,
    $win_to                 AS TO_DATE
FROM telecasts
QUALIFY RNK <= $p_top_n
ORDER BY RNK;